# ADU + NSU + VDU RL Agent — Architecture Comparison Notebook

## Crude Distillation Unit (ADU + NSU + VDU) Optimisation via Reinforcement Learning

This notebook benchmarks **three RL architectures** on the ADU+NSU+VDU Gymnasium environment, capturing the same metrics dashboard visible in the web application:

| Metric group | Metrics captured |
|---|---|
| **Performance** | Episode reward, avg reward (100-ep), best reward |
| **Critic / Actor losses** | `train/critic_loss`, `train/actor_loss` |
| **Entropy / Alpha** | Policy entropy, temperature coefficient α |
| **Q-values** | Mean Q-value sampled from replay buffer |
| **Gradient norms** | Actor ∇ and Critic ∇ (L2 norm) |
| **Replay buffer** | Buffer size and % utilisation |
| **Action distribution** | Mean ± std of each of the 10 ADU+NSU+VDU action dimensions |

### Architectures compared
- **SAC** — Soft Actor-Critic (off-policy, max-entropy framework) ← baseline
- **TD3** — Twin Delayed DDPG (off-policy, deterministic policy, clipped-double Q)
- **PPO** — Proximal Policy Optimisation (on-policy, clipped surrogate objective)

### Environment Setup

In [23]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parents[1] if "__file__" in dir() else Path("D:/github/Distillation-column-agent")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)

Project root: D:\github\Distillation-column-agent
Python: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]


### Imports

In [24]:
import time, json, warnings
from datetime import datetime
from typing import Any, Callable, Optional
from copy import deepcopy

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
import pandas as pd

from IPython.display import display, clear_output, HTML

from stable_baselines3 import SAC, PPO, TD3
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import DummyVecEnv

warnings.filterwarnings("ignore")
plt.style.use("dark_background")

# Colour palettes for easier visual distinction in plots
PALETTE = {
    "SAC":  "#3b82f6",   # blue
    "TD3":  "#22c55e",   # green
    "PPO":  "#a855f7",   # purple
}
METRIC_COLORS = {
    "critic": "#f97316",  # orange
    "actor":  "#ef4444",  # red
    "q":      "#06b6d4",  # cyan
    "entropy":"#eab308",  # yellow
    "alpha":  "#ec4899",  # pink
    "actor_grad":  "#3b82f6",
    "critic_grad": "#f97316",
    "buffer": "#64748b",  # slate
}

ACTION_KEYS = [
    "reflux_ratio", "hn_draw_temp", "sko_draw_temp",
    "ld_draw_temp", "hd_draw_temp", "atmos_reboiler_temp",
    "nsu_reflux_ratio", "nsu_reboiler_temp",
    "vac_reflux_ratio", "vac_reboiler_temp",
    "vac_diesel_draw_temp", "vgo_draw_temp",
]
ACTION_COLORS = [
    "#3b82f6","#22c55e","#a855f7","#f97316","#ef4444",
    "#06b6d4","#eab308","#ec4899","#64748b","#78350f",
    "#10b981","#f43f5e",
]

print("All imports successful!")


All imports successful!


---
## Section 1 — DWSIM Connection & Environment

The `CDUEnvironment` wraps the DWSIM flowsheet (`main_sim.dwxmz`) via `DWSIMBridge`.
The flowsheet has three columns: an Atmospheric Distillation Unit (ADU), a Naphtha Stabilizer (NSU), and a Vacuum Distillation Unit (VDU), producing 13 product streams.
D95% distillation temperature is estimated for each product from component compositions and Normal Boiling Points.

### DWSIM Connection Test

In [25]:
import importlib

# This is added to force-reload backend modules to pick up any edits made since kernel start
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("backend."):
        try:
            importlib.reload(sys.modules[mod_name])
        except Exception:
            pass  # some sub-modules may fail; OK

# USE_DWSIM=True — test the corrected bridge against main_sim.dwxmz
# This is a fail-safe mechanism added, in case DWSim simulation is not loaded properly by the bridge script
USE_DWSIM = True



### -------- Important!!! This filename/path needed to be updated to match the actual flowsheet file --------
FLOWSHEET_PATH = str(PROJECT_ROOT / "Sim_models" / "main_sim.dwxmz")

if USE_DWSIM:
    ### This part is dependant on the dwsim_bridge script which is going to be common for both the notebook and the application to maintain single source of truth
    from backend.core.dwsim_bridge import DWSIMBridge, PRODUCT_STREAMS, VAC_COLUMN_NAME
    print(f"PRODUCT_STREAMS check: {len(PRODUCT_STREAMS)} products")
    for k, v in PRODUCT_STREAMS.items():
        print(f"  {k} → {v}")
    print("\nTesting DWSIM bridge with corrected stream names …")
    try:
        bridge = DWSIMBridge(FLOWSHEET_PATH)
        bridge.load()
        state = bridge.get_column_state()
        print("\n DWSIM connected — column state snapshot:")
        for k, v in sorted(state.items()):
            print(f"  {k:<35s} = {v:.4f}")

        # Show D95% estimates
        d95 = bridge.get_d95_all_products()
        print("\n D95% estimates (°C):")
        for prod, temp in d95.items():
            print(f"  {prod:<25s} = {temp:.1f}")

        bridge.close()
        del bridge
        print("\n All stream / column names resolved correctly.")
    except Exception as e:
        print(f"\n DWSIM connection failed: {e}")
        print("  Falling back to mock mode for training.")
        USE_DWSIM = False
else:
    print("Mock mode — DWSIM bridge skipped.")
    print("All 13 product streams (ADU: 5, NSU: 3, VDU: 5) will be simulated stochastically.")
    print(f"Flowsheet path (for reference): {FLOWSHEET_PATH}")

2026-03-07 13:08:33.861 | INFO     | backend.core.dwsim_bridge:_bootstrap_clr:61 - CLR bootstrapped with DWSIM assemblies


PRODUCT_STREAMS check: 13 products
  Uncondensed_Gas → Uncondensed_Gas
  Heavy_Naphtha → Heavy_Naphtha
  SKO → SKO
  Light_Gas_Oil → Light_Gas_Oil
  Heavy_Gas_Oil → Heavy_Gas_Oil
  StabOffGas → StabOffGas
  LPG → LPG
  SRN → SRN
  Offgas → Offgas
  Vacuum_Diesel → Vacuum_Diesel
  Vacuum_Gas_Oil → Vacuum_Gas_Oil
  Hotwell_Oil → Hotwell_Oil
  Vac_residue → Vac_residue

Testing DWSIM bridge with corrected stream names …


2026-03-07 13:08:34.427 | INFO     | backend.core.dwsim_bridge:__init__:226 - DWSIMBridge created for D:\github\Distillation-column-agent\Sim_models\main_sim.dwxmz
2026-03-07 13:08:35.606 | INFO     | backend.core.dwsim_bridge:load:234 - Flowsheet loaded
2026-03-07 13:08:35.645 | INFO     | backend.core.dwsim_bridge:close:247 - DWSIM resources released



 DWSIM connected — column state snapshot:
  bottom_pressure                     = 210.0000
  bottom_temperature                  = 340.5717
  condenser_duty                      = 59.5700
  d95_Heavy_Gas_Oil                   = 221.9177
  d95_Heavy_Naphtha                   = 221.6489
  d95_Hotwell_Oil                     = 278.6996
  d95_LPG                             = 164.0072
  d95_Light_Gas_Oil                   = 221.8421
  d95_Offgas                          = 274.8235
  d95_SKO                             = 221.8212
  d95_SRN                             = 197.9767
  d95_StabOffGas                      = 112.3745
  d95_Uncondensed_Gas                 = 150.3874
  d95_Vac_residue                     = 619.1681
  d95_Vacuum_Diesel                   = 332.9296
  d95_Vacuum_Gas_Oil                  = 336.9737
  feed_flow_rate                      = 500000.0000
  feed_temperature                    = 360.0000
  flow_Heavy_Gas_Oil                  = 16096.5169
  flow_Heavy_Naphtha  

### Building ADU + NSU + VDU Gymnasium Environment

In [26]:
from backend.core.rl_environment import (
    CDUEnvironment, ACTION_SCALES, ACTION_HARD_LIMITS, ACTION_KEYS,
    PRODUCT_KEYS, DEFAULT_PRICES, D95_SPECS,
    WARMUP_INITIAL, WARMUP_END_FRAC,
    TOL_INITIAL, TOL_MIN, TOL_MAX, TOL_ESCALATION_FACTOR, TOL_RELAX_FACTOR,
)

def make_env(use_mock: bool = True, curriculum: float = 1.0,
             max_ep_steps: int = 50) -> CDUEnvironment:
    '''Factory that returns a configured CDUEnvironment.'''
    return CDUEnvironment(
        flowsheet_path=FLOWSHEET_PATH if not use_mock else None,
        prices=DEFAULT_PRICES.copy(),
        max_steps=max_ep_steps,
        curriculum_level=curriculum,
        use_mock=use_mock,
    )

# Quick sanity check
_env = make_env(use_mock=True)
obs, info = _env.reset()
print(f"Observation space  : {_env.observation_space}")
print(f"Action space       : {_env.action_space}")
print(f"Obs shape          : {obs.shape}  (min={obs.min():.3f}, max={obs.max():.3f})")
print(f"Action dimensions  : {len(ACTION_KEYS)} -> {ACTION_KEYS}")

# One random step
sample_action = _env.action_space.sample()
obs2, reward, terminated, truncated, info2 = _env.step(sample_action)
print(f"\nStep reward        : {reward:.6f}")
print(f"Warmup factor      : {_env._warmup_factor():.3f}")
print(f"State keys         : {list(info2['state'].keys())[:6]} ...")
_env.close()
print("\n Environment OK (mock mode) -- 16-dim action space with warmup")

2026-03-07 13:08:35.654 | INFO     | backend.core.rl_environment:__init__:200 - CDUEnvironment created | mock=True | max_steps=200 | curriculum=1.00


Observation space  : Box(0.0, 1.0, (37,), float32)
Action space       : Box(-1.0, 1.0, (12,), float32)
Obs shape          : (37,)  (min=0.000, max=0.852)
Step reward        : 0.986731
State keys         : ['flow_Uncondensed_Gas', 'temp_Uncondensed_Gas', 'd95_Uncondensed_Gas', 'flow_Heavy_Naphtha', 'temp_Heavy_Naphtha', 'd95_Heavy_Naphtha'] …

 Environment OK (mock mode)


---
## Section 2 — Detailed Metrics Callback

`NotebookMetricsCallback` is a self-contained replica of the application's `ProgressCallback`.  
It hooks into SB3's training loop to capture all 12 metric channels that the web dashboard will show.

### Notebook Metrics Callback

In [27]:
class NotebookMetricsCallback(BaseCallback):
    """
    Comprehensive SB3 callback that mirrors the web application's metrics dashboard.

    Captured channels
    -----------------
    Performance : episode_reward, avg_reward (100-ep), best_reward
    Losses      : critic_loss, actor_loss
    Entropy   : entropy (policy), ent_coef
    Q-value     : mean_q_value (sampled from replay buffer)
    Grad norms  : actor_grad_norm, critic_grad_norm
    Buffer      : replay_buffer_size, capacity, pct_full
    Action dist : means + stds of each action dimension
    """

    def __init__(self, log_interval: int = 200, verbose: int = 0):
        super().__init__(verbose)
        self.log_interval = log_interval

        # Episode tracking
        self.episode_rewards: list[float] = []
        self._cur_ep_reward: float = 0.0
        self.episode_count: int = 0
        self.best_reward: float = float("-inf")

        # Raw action history for distribution analysis
        self._action_buf: list[list[float]] = []

        # Full history (one dict per log_interval steps)
        self.history: list[dict] = []

    # SB3 hook

    def _on_step(self) -> bool:
        reward = self.locals.get("rewards", [0])[0]
        self._cur_ep_reward += float(reward)

        # Buffer latest action
        action = self.locals.get("actions")
        if action is not None:
            a = action[0]
            self._action_buf.append(a.tolist() if hasattr(a, "tolist") else list(a))
            if len(self._action_buf) > 1000:
                self._action_buf = self._action_buf[-1000:]

        # Episode end
        if self.locals.get("dones", [False])[0]:
            self.episode_rewards.append(self._cur_ep_reward)
            self.episode_count += 1
            if self._cur_ep_reward > self.best_reward:
                self.best_reward = self._cur_ep_reward
            self._cur_ep_reward = 0.0

        # Periodic log
        if self.num_timesteps % self.log_interval == 0:
            self.history.append(self._snapshot())

        return True

    # Metric collection helper

    def _snapshot(self) -> dict:
        """Collect a full metrics snapshot at the current step."""
        avg = float(np.mean(self.episode_rewards[-100:])) if self.episode_rewards else 0.0
        m: dict[str, Any] = {
            "step":         self.num_timesteps,
            "episode":      self.episode_count,
            "episode_reward": self.episode_rewards[-1] if self.episode_rewards else 0.0,
            "avg_reward":   avg,
            "best_reward":  self.best_reward,
        }

        # SB3 logged losses / ent_coef
        if hasattr(self.model, "logger") and hasattr(self.model.logger, "name_to_value"):
            lv = self.model.logger.name_to_value
            m["critic_loss"]  = float(lv.get("train/critic_loss",  0))
            m["actor_loss"]   = float(lv.get("train/actor_loss",   0))
            m["ent_coef"]     = float(lv.get("train/ent_coef",     0))
            m["ent_coef_loss"]= float(lv.get("train/ent_coef_loss",0))
            m["n_updates"]    = int(  lv.get("train/n_updates",    0))
            # PPO reports these differently
            m["value_loss"]   = float(lv.get("train/value_loss",   0))
            m["policy_gradient_loss"] = float(lv.get("train/policy_gradient_loss", 0))
            m["approx_kl"]    = float(lv.get("train/approx_kl",    0))
            m["clip_fraction"]= float(lv.get("train/clip_fraction", 0))

        # Replay buffer (off-policy only)
        if getattr(self.model, "replay_buffer", None) is not None:
            buf = self.model.replay_buffer
            sz  = int(buf.size())
            cap = int(buf.buffer_size)
            m["replay_buffer_size"]     = sz
            m["replay_buffer_capacity"] = cap
            m["replay_buffer_pct"]      = round(sz / cap * 100, 1)

            # Mean Q-value via critic on sampled batch
            if sz > getattr(self.model, "batch_size", 64):
                try:
                    n = min(64, sz)
                    rd = buf.sample(n)
                    with torch.no_grad():
                        qs = self.model.critic(rd.observations, rd.actions)
                        if isinstance(qs, (list, tuple)):
                            all_q = torch.cat([q.flatten() for q in qs])
                        else:
                            all_q = qs.flatten()
                    m["mean_q_value"] = round(float(all_q.mean().item()), 4)
                except Exception:
                    pass

            # Policy entropy via actor log-prob
            if sz > getattr(self.model, "batch_size", 64):
                try:
                    n = min(64, sz)
                    rd = buf.sample(n)
                    with torch.no_grad():
                        _, log_prob = self.model.actor.action_log_prob(rd.observations)
                    m["entropy"] = round(float(-log_prob.mean().item()), 4)
                except Exception:
                    pass

        # Gradient norms
        for net_name in ("actor", "critic", "policy"):
            net = getattr(self.model, net_name, None)
            if net is None:
                continue
            try:
                total = sum(
                    p.grad.data.norm(2).item() ** 2
                    for p in net.parameters()
                    if p.grad is not None
                )
                m[f"{net_name}_grad_norm"] = round(float(total ** 0.5), 6)
            except Exception:
                pass

        # Action distribution
        if len(self._action_buf) > 10:
            arr = np.array(self._action_buf[-200:])
            m["action_distribution"] = {
                "means": np.round(arr.mean(axis=0), 4).tolist(),
                "stds":  np.round(arr.std(axis=0),  4).tolist(),
                "names": ACTION_KEYS,
            }

        # Sanitise: numpy → native Python
        return _sanitise(m)


def _sanitise(val: Any) -> Any:
    """Recursively convert numpy/torch scalars to JSON-serialisable Python types."""
    if val is None:
        return None
    if isinstance(val, (bool, int, str)):
        return val
    if isinstance(val, float):
        return 0.0 if (np.isinf(val) or np.isnan(val)) else val
    if isinstance(val, np.generic):
        v = val.item()
        return 0.0 if isinstance(v, float) and (np.isinf(v) or np.isnan(v)) else v
    if isinstance(val, np.ndarray):
        return val.tolist()
    if isinstance(val, dict):
        return {k: _sanitise(v) for k, v in val.items()}
    if isinstance(val, (list, tuple)):
        return [_sanitise(v) for v in val]
    if hasattr(val, "item"):
        try:
            return float(val.item())
        except Exception:
            pass
    return val

In [29]:
## -- Pre-Training Summary: Pricing / Reward / D95 / Actions / Warmup

import numpy as np
from backend.core.rl_environment import (
    DEFAULT_PRICES, D95_SPECS, ACTION_SCALES, ACTION_HARD_LIMITS, ACTION_KEYS,
    PRODUCT_KEYS, WARMUP_INITIAL, WARMUP_END_FRAC,
    TOL_INITIAL, TOL_MIN, TOL_MAX, TOL_ESCALATION_FACTOR, TOL_RELAX_FACTOR,
)

SEP  = chr(0x2500) * 72
SEP2 = chr(0x2550) * 72

# -- 1. Product Prices
print(SEP2)
print("  PRODUCT PRICES  ($/kg)")
print(SEP2)
print(f"  {'Product':<25s}  {'Price $/kg':>10s}  Column")
print(SEP)
COLUMN_TAG = {
    "Uncondensed_Gas": "ADU", "Heavy_Naphtha": "ADU", "SKO": "ADU",
    "Light_Gas_Oil":   "ADU", "Heavy_Gas_Oil": "ADU",
    "StabOffGas":      "NSU", "LPG":           "NSU", "SRN": "NSU",
    "Offgas":          "VDU", "Vacuum_Diesel": "VDU", "Vacuum_Gas_Oil": "VDU",
    "Hotwell_Oil":     "VDU", "Vac_residue":   "VDU",
    "Feed_Crude":      "Feed",
}
for prod, price in DEFAULT_PRICES.items():
    tag    = COLUMN_TAG.get(prod, "")
    marker = "  <- feed cost" if prod == "Feed_Crude" else ""
    print(f"  {prod:<25s}  {price:>10.2f}  {tag}{marker}")

# -- 2. D95% Specification Limits
print(f"\n{SEP2}")
print("  D95% SPECIFICATION LIMITS  (C)  --  penalty = 2.0 x excess C")
print(SEP2)
print(f"  {'Product':<25s}  {'D95 limit (C)':>14s}  Penalty rate")
print(SEP)
for prod, spec in D95_SPECS.items():
    print(f"  {prod:<25s}  {spec:>14.1f}  2.0 $/C above limit")
print()
print("  No D95 spec: Uncondensed_Gas, StabOffGas, LPG, Offgas, Hotwell_Oil, SRN, Vac_residue")

# -- 3. Reward Formula
print(f"\n{SEP2}")
print("  REWARD FORMULA")
print(SEP2)
print("  reward = ( sum(flow_i * price_i) - feed_cost - d95_penalty - safety_penalty ) / 100.0")

# -- 4. Action Scales, Hard Limits
print(f"\n{SEP2}")
print("  ACTION SPACE  (16 dimensions)  --  MAX delta PER STEP  &  HARD CLAMP")
print(SEP2)
print(f"  {'Action key':<25s}  {'+-d/step':>9s}  {'Hard min':>9s}  {'Hard max':>9s}  {'Unit':>6s}  Col")
print(SEP)
_COL = {
    "reflux_ratio": "ADU", "hn_draw_temp": "ADU", "sko_draw_temp": "ADU",
    "ld_draw_temp": "ADU", "hd_draw_temp": "ADU", "atmos_reboiler_temp": "ADU",
    "atmos_top_pressure": "ADU", "atmos_dp": "ADU",
    "nsu_reflux_ratio": "NSU", "nsu_reboiler_temp": "NSU",
    "vac_reflux_ratio": "VDU", "vac_reboiler_temp": "VDU",
    "vac_diesel_draw_temp": "VDU", "vgo_draw_temp": "VDU",
    "vac_top_pressure": "VDU", "vac_dp": "VDU",
}
for k in ACTION_KEYS:
    scale  = ACTION_SCALES[k]
    lo, hi = ACTION_HARD_LIMITS[k]
    unit   = "kPa" if ("pressure" in k or k.endswith("_dp")) else ("C" if "temp" in k else "--")
    col    = _COL.get(k, "")
    print(f"  {k:<25s}  {scale:>9.2f}  {lo:>9.1f}  {hi:>9.1f}  {unit:>6s}  {col}")

# -- 5. Progressive Warmup
print(f"\n{SEP2}")
print("  PROGRESSIVE ACTION WARMUP")
print(SEP2)
print(f"  Initial scale at step 1  :  {WARMUP_INITIAL*100:.0f}% of full +-delta")
print(f"  Full scale reached at    :  {WARMUP_END_FRAC*100:.0f}% of max_steps")

# -- 6. Adaptive Solver Tolerance
print(f"\n{SEP2}")
print("  ADAPTIVE SOLVER TOLERANCE")
print(SEP2)
print(f"  Initial: {TOL_INITIAL}  |  Range: [{TOL_MIN}, {TOL_MAX}]")
print(f"  Escalate x{TOL_ESCALATION_FACTOR} on failure (retry)  |  Relax x{TOL_RELAX_FACTOR} on success")
print(SEP)

═════════════════════════════════════════════════════════════════
  PRODUCT PRICES  ($/kg)
═════════════════════════════════════════════════════════════════
  Product                    Price $/kg  Column
─────────────────────────────────────────────────────────────────
  Uncondensed_Gas                  0.30  ADU
  Heavy_Naphtha                    0.60  ADU
  SKO                              0.75  ADU
  Light_Gas_Oil                    0.70  ADU
  Heavy_Gas_Oil                    0.55  ADU
  StabOffGas                       0.20  NSU
  LPG                              0.55  NSU
  SRN                              0.65  NSU
  Offgas                           0.15  VDU
  Vacuum_Diesel                    0.52  VDU
  Vacuum_Gas_Oil                   0.45  VDU
  Hotwell_Oil                      0.30  VDU
  Vac_residue                      0.25  VDU
  Feed_Crude                       0.45  Feed  ← feed cost

═════════════════════════════════════════════════════════════════
  D95% SPECIFICATI

In [ ]:
## -- ACTION SPACE & WARMUP VISUALIZATION --

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from backend.core.rl_environment import (
    ACTION_SCALES, ACTION_HARD_LIMITS, ACTION_KEYS,
    WARMUP_INITIAL, WARMUP_END_FRAC,
)

MAX_EP_STEPS = 50

# -- Fig 1: Warmup curve --
steps = np.arange(1, MAX_EP_STEPS + 1)
warmup = np.minimum(1.0, WARMUP_INITIAL + (1.0 - WARMUP_INITIAL) * (steps / MAX_EP_STEPS) / WARMUP_END_FRAC)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.plot(steps, warmup * 100, color="#2196F3", linewidth=2.5)
ax.fill_between(steps, 0, warmup * 100, alpha=0.15, color="#2196F3")
ax.axhline(100, color="grey", linestyle="--", linewidth=0.8, label="Full scale")
ax.set_xlabel("Step within episode", fontsize=12)
ax.set_ylabel("Action scale (%)", fontsize=12)
ax.set_title("Progressive Action Warmup per Episode", fontsize=13, fontweight="bold")
ax.set_ylim(0, 115)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Annotate key points
step_50pct = int(MAX_EP_STEPS * 0.5)
wf_50 = min(1.0, WARMUP_INITIAL + (1.0 - WARMUP_INITIAL) * 0.5 / WARMUP_END_FRAC)
ax.annotate(f"Step 1: {warmup[0]*100:.1f}%",
            xy=(1, warmup[0]*100), xytext=(5, 25),
            arrowprops=dict(arrowstyle="->", color="red"), fontsize=10, color="red")
ax.annotate(f"Step {step_50pct}: {wf_50*100:.0f}%",
            xy=(step_50pct, wf_50*100), xytext=(step_50pct+3, wf_50*100 - 15),
            arrowprops=dict(arrowstyle="->", color="green"), fontsize=10, color="green")

# -- Fig 2: Effective delta per action at different episode stages --
ax2 = axes[1]
n_actions = len(ACTION_KEYS)
y_pos = np.arange(n_actions)

warmup_stages = {
    f"Step 1  ({warmup[0]*100:.0f}%)":  warmup[0],
    f"Step {MAX_EP_STEPS//2} ({warmup[MAX_EP_STEPS//2 - 1]*100:.0f}%)": warmup[MAX_EP_STEPS//2 - 1],
    f"Step {MAX_EP_STEPS} (100%)": 1.0,
}
colors = ["#FF9800", "#4CAF50", "#2196F3"]
bar_height = 0.25

for idx, (label, wf) in enumerate(warmup_stages.items()):
    deltas = [ACTION_SCALES[k] * wf for k in ACTION_KEYS]
    offset = (idx - 1) * bar_height
    ax2.barh(y_pos + offset, deltas, bar_height, label=label, color=colors[idx], alpha=0.8)

ax2.set_yticks(y_pos)
ax2.set_yticklabels([k.replace("_", "\n", 1) if len(k) > 18 else k for k in ACTION_KEYS], fontsize=8)
ax2.set_xlabel("Effective +/-delta per step (physical units)", fontsize=11)
ax2.set_title("Action Magnitude at Different Episode Stages", fontsize=13, fontweight="bold")
ax2.legend(fontsize=9, loc="lower right")
ax2.grid(True, axis="x", alpha=0.3)
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

# -- Fig 3: Action space hard limits (operating window) --
fig2, ax3 = plt.subplots(figsize=(14, 7))

labels = ACTION_KEYS
lo_vals = [ACTION_HARD_LIMITS[k][0] for k in labels]
hi_vals = [ACTION_HARD_LIMITS[k][1] for k in labels]
ranges  = [hi - lo for lo, hi in zip(lo_vals, hi_vals)]
y_pos   = np.arange(len(labels))

col_colors = {"ADU": "#E91E63", "NSU": "#9C27B0", "VDU": "#00BCD4"}
_COL_MAP = {
    "reflux_ratio": "ADU", "hn_draw_temp": "ADU", "sko_draw_temp": "ADU",
    "ld_draw_temp": "ADU", "hd_draw_temp": "ADU", "atmos_reboiler_temp": "ADU",
    "atmos_top_pressure": "ADU", "atmos_dp": "ADU",
    "nsu_reflux_ratio": "NSU", "nsu_reboiler_temp": "NSU",
    "vac_reflux_ratio": "VDU", "vac_reboiler_temp": "VDU",
    "vac_diesel_draw_temp": "VDU", "vgo_draw_temp": "VDU",
    "vac_top_pressure": "VDU", "vac_dp": "VDU",
}
bar_colors = [col_colors[_COL_MAP[k]] for k in labels]

ax3.barh(y_pos, ranges, left=lo_vals, color=bar_colors, alpha=0.7, edgecolor="white", height=0.7)

for i, k in enumerate(labels):
    lo, hi = ACTION_HARD_LIMITS[k]
    mid = (lo + hi) / 2
    ax3.text(mid, i, f"[{lo:.0f} - {hi:.0f}]", ha="center", va="center", fontsize=8, fontweight="bold")

ax3.set_yticks(y_pos)
ax3.set_yticklabels(labels, fontsize=9)
ax3.set_xlabel("Physical value range", fontsize=12)
ax3.set_title("Action Space: Hard Clamp Limits per Parameter", fontsize=14, fontweight="bold")
ax3.invert_yaxis()
ax3.grid(True, axis="x", alpha=0.3)

legend_patches = [Patch(color=c, label=col, alpha=0.7) for col, c in col_colors.items()]
ax3.legend(handles=legend_patches, fontsize=11, loc="lower right")

plt.tight_layout()
plt.show()

print("\n 16-dim action space with progressive warmup ready for training.")

---
## Section 3 -- Training Each Architecture

We train all three on identical environments with **16-dim action space** (including column pressures & DP), **progressive warmup**, and **adaptive solver tolerance**.

| Hyperparameter | Value |
|---|---|
| Total timesteps | 500 |
| Max episode steps | 50 |
| Learning starts (SAC/TD3) | 100 |
| Batch size | 64 |
| Learning rate | 3e-4 |
| Discount factor | 0.99 |
| Buffer size (SAC/TD3) | 10,000 |
| Action warmup | 5% -> 100% over 70% of episode |
| Solver tolerance | 0.5 (adaptive: 0.1-100) |
| Curriculum | Full difficulty (level 1.0) |

### Training Configuration & Utility

In [ ]:
### Adjustable training configurations
TOTAL_TIMESTEPS = 500
MAX_EP_STEPS    = 50        # shorter episodes -> more resets
LEARNING_STARTS = 100       # MUST be < TOTAL_TIMESTEPS
LOG_INTERVAL    = 10
BATCH_SIZE      = 64
LEARNING_RATE   = 3e-4
GAMMA           = 0.99

USE_MOCK = False


def train_agent(
    algo_class,
    algo_name: str,
    total_timesteps: int = TOTAL_TIMESTEPS,
    extra_kwargs: Optional[dict] = None,
) -> tuple[Any, NotebookMetricsCallback, float]:
    '''Train a single SB3 agent and return (model, callback, elapsed).'''
    print(f"\n{'='*60}")
    print(f" Training {algo_name}  ({total_timesteps:,} steps, "
          f"max_ep={MAX_EP_STEPS}, batch={BATCH_SIZE})")
    print(f"{'='*60}")

    vec_env = DummyVecEnv([lambda: make_env(use_mock=USE_MOCK, max_ep_steps=MAX_EP_STEPS)])

    base_kwargs = dict(
        policy="MlpPolicy",
        env=vec_env,
        learning_rate=LEARNING_RATE,
        gamma=GAMMA,
        verbose=0,
        device="auto",
    )

    if algo_class in (SAC, TD3):
        base_kwargs["batch_size"] = BATCH_SIZE
        base_kwargs["buffer_size"] = 10_000
        base_kwargs["learning_starts"] = LEARNING_STARTS
        if algo_class is SAC:
            base_kwargs["ent_coef"] = "auto"
    elif algo_class is PPO:
        base_kwargs["n_steps"]    = 512
        base_kwargs["batch_size"] = BATCH_SIZE
        base_kwargs["n_epochs"]   = 10
        base_kwargs["clip_range"] = 0.2

    if extra_kwargs:
        base_kwargs.update(extra_kwargs)

    model = algo_class(**base_kwargs)
    cb = NotebookMetricsCallback(log_interval=LOG_INTERVAL)

    t0 = time.time()
    model.learn(total_timesteps=total_timesteps, callback=cb, progress_bar=False)
    elapsed = time.time() - t0

    episodes = cb.episode_count
    best     = cb.best_reward
    avg      = float(np.mean(cb.episode_rewards[-100:])) if cb.episode_rewards else 0.0

    print(f"  Done in {elapsed:.1f}s | Episodes: {episodes} | "
          f"Best: {best:.4f} | Avg(100): {avg:.4f}")

    vec_env.close()
    return model, cb, elapsed


RESULTS: dict[str, dict] = {}
print(f"Training utility ready  (TOTAL_TIMESTEPS={TOTAL_TIMESTEPS} | "
      f"MAX_EP_STEPS={MAX_EP_STEPS} | LEARNING_STARTS={LEARNING_STARTS} | "
      f"BATCH_SIZE={BATCH_SIZE} | USE_MOCK={USE_MOCK})")

Training utility ready   (TOTAL_TIMESTEPS = 500 | USE_MOCK = False )


### Train Soft Actor-Critic

SAC is the production default — max-entropy framework with automatic α tuning.

In [ ]:
sac_model, sac_cb, sac_time = train_agent(SAC, "SAC")
RESULTS["SAC"] = {"model": sac_model, "cb": sac_cb, "time": sac_time}

# Quick peek at final metrics
h = sac_cb.history
if h:
    last = h[-1]
    print(f"\n  Final snapshot (step {last['step']}):")
    for k in ("avg_reward","best_reward","critic_loss","actor_loss",
              "mean_q_value","entropy","ent_coef","actor_grad_norm",
              "critic_grad_norm","replay_buffer_pct"):
        print(f"    {k:<25s} = {last.get(k, 'n/a')}")


 Training SAC  (500 steps)


2026-03-07 13:00:57.670 | INFO     | backend.core.dwsim_bridge:__init__:226 - DWSIMBridge created for D:\github\Distillation-column-agent\Sim_models\main_sim.dwxmz
2026-03-07 13:00:57.670 | INFO     | backend.core.rl_environment:__init__:200 - CDUEnvironment created | mock=False | max_steps=200 | curriculum=1.00
2026-03-07 13:00:57.676 | INFO     | backend.core.rl_environment:reset:218 - =======================================================
 Episode 2 START — reloading flowsheet to baseline
2026-03-07 13:00:58.752 | INFO     | backend.core.dwsim_bridge:load:234 - Flowsheet loaded
2026-03-07 13:00:58.754 | INFO     | backend.core.dwsim_bridge:get_current_operating_point:559 - Current operating point read from simulation: {'reflux_ratio': 5.0, 'hn_draw_temp': 242.05464319653697, 'sko_draw_temp': 244.30785716119203, 'ld_draw_temp': 246.80129542263, 'hd_draw_temp': 247.95657528720608, 'atmos_reboiler_temp': 365.0, 'nsu_reflux_ratio': 5.0, 'nsu_reboiler_temp': 155.0, 'vac_reflux_ratio': 5

KeyboardInterrupt: 

### Train Twin Delayed DDPG

TD3 uses deterministic policy, target-policy smoothing, and delayed actor updates.
No entropy term — policy_freq controls actor update delay.

In [ ]:
td3_model, td3_cb, td3_time = train_agent(
    TD3, "TD3",
    extra_kwargs={"policy_delay": 2, "target_policy_noise": 0.2, "target_noise_clip": 0.5},
)
RESULTS["TD3"] = {"model": td3_model, "cb": td3_cb, "time": td3_time}

h = td3_cb.history
if h:
    last = h[-1]
    print(f"\n  Final snapshot (step {last['step']}):")
    for k in ("avg_reward","best_reward","critic_loss","actor_loss",
              "mean_q_value","actor_grad_norm","critic_grad_norm","replay_buffer_pct"):
        print(f"    {k:<25s} = {last.get(k, 'n/a')}")

### Train PPO (Proximal Policy Optimisation)

PPO is on-policy — no replay buffer, so Q-value / buffer metrics are N/A.
Instead it exposes value_loss, policy_gradient_loss, approx_kl, clip_fraction.

In [ ]:
ppo_model, ppo_cb, ppo_time = train_agent(
    PPO, "PPO",
    extra_kwargs={"ent_coef": 0.01},   # explicit entropy bonus for exploration
)
RESULTS["PPO"] = {"model": ppo_model, "cb": ppo_cb, "time": ppo_time}

h = ppo_cb.history
if h:
    last = h[-1]
    print(f"\n  Final snapshot (step {last['step']}):")
    for k in ("avg_reward","best_reward","value_loss","policy_gradient_loss",
              "approx_kl","clip_fraction","policy_grad_norm"):
        print(f"    {k:<30s} = {last.get(k, 'n/a')}")

---
## Section 4 — Performance Metrics Comparison

The cells below reproduce the full dashboard — one chart panel per metric group.  
Charts use the same colour coding as the web application.

### Chart Helper Utilities

In [ ]:
def _steps(cb: NotebookMetricsCallback) -> np.ndarray:
    return np.array([m["step"] for m in cb.history])

def _get(cb: NotebookMetricsCallback, key: str) -> np.ndarray:
    return np.array([m.get(key, np.nan) for m in cb.history], dtype=float)

def _smooth(arr: np.ndarray, w: int = 5) -> np.ndarray:
    """Simple moving average for smoother loss curves."""
    if len(arr) < w:
        return arr
    kernel = np.ones(w) / w
    pad = np.pad(arr, (w//2, w//2), mode="edge")
    return np.convolve(pad, kernel, mode="valid")[: len(arr)]

def _style_ax(ax, title: str, xlabel="Step", ylabel=""):
    ax.set_title(title, color="white", fontsize=11, pad=8)
    ax.set_xlabel(xlabel, color="#94a3b8", fontsize=9)
    ax.set_ylabel(ylabel, color="#94a3b8", fontsize=9)
    ax.tick_params(colors="#94a3b8", labelsize=8)
    ax.spines[:].set_color("#334155")
    ax.xaxis.set_major_locator(MaxNLocator(5, integer=True))
    ax.grid(True, linestyle="--", alpha=0.2, color="#334155")

FIG_BG = "#0f172a"   # page background
PANEL_BG = "#1e293b" # card background

def _fig(rows=1, cols=1, figsize=None, title=None):
    fs = figsize or (14, 4 * rows)
    fig, axes = plt.subplots(rows, cols, figsize=fs, facecolor=FIG_BG,
                              squeeze=False)
    fig.patch.set_facecolor(FIG_BG)
    for ax in axes.flat:
        ax.set_facecolor(PANEL_BG)
    if title:
        fig.suptitle(title, color="white", fontsize=13, y=0.98)
    return fig, axes

print("Chart helpers ready ")

### Chart: Reward Curves

In [ ]:
fig, axes = _fig(1, 2, figsize=(15, 5), title="Performance Metrics — Episode Reward")

ax_avg, ax_best = axes[0]

for name, res in RESULTS.items():
    cb   = res["cb"]
    sx   = _steps(cb)
    col  = PALETTE[name]
    avg  = _get(cb, "avg_reward")
    best = _get(cb, "best_reward")
    ax_avg.plot(sx, avg,  color=col, lw=1.8, label=name, alpha=0.9)
    ax_best.plot(sx, best, color=col, lw=1.8, label=name, alpha=0.9)

_style_ax(ax_avg,  "Avg Reward (100-episode rolling)", ylabel="Reward")
_style_ax(ax_best, "Best Reward (cumulative)",          ylabel="Reward")
for ax in (ax_avg, ax_best):
    ax.legend(fontsize=9, facecolor="#1e293b", edgecolor="#334155",
              labelcolor="white")

plt.tight_layout()
plt.show()

### Chart: Critic & Actor Loss

Off-policy (SAC, TD3) expose train/critic_loss + train/actor_loss

On-policy  (PPO)      exposes value_loss + policy_gradient_loss

In [ ]:
fig, axes = _fig(2, 2, figsize=(15, 9), title="Loss Curves")
ax_c, ax_a, ax_pv, ax_ppg = axes[0][0], axes[0][1], axes[1][0], axes[1][1]

# Critic loss — SAC & TD3
for name in ("SAC", "TD3"):
    if name not in RESULTS: continue
    cb  = RESULTS[name]["cb"]
    sx  = _steps(cb)
    val = _smooth(_get(cb, "critic_loss"))
    mask = ~np.isnan(val) & (val > 0)
    if mask.any():
        ax_c.plot(sx[mask], val[mask], color=PALETTE[name], lw=1.5, label=name)

# Actor loss — SAC & TD3
for name in ("SAC", "TD3"):
    if name not in RESULTS: continue
    cb  = RESULTS[name]["cb"]
    sx  = _steps(cb)
    val = _smooth(_get(cb, "actor_loss"))
    mask = ~np.isnan(val)
    ax_a.plot(sx[mask], val[mask], color=PALETTE[name], lw=1.5, label=name)

# PPO value loss
if "PPO" in RESULTS:
    cb  = RESULTS["PPO"]["cb"]
    sx  = _steps(cb)
    val = _smooth(_get(cb, "value_loss"))
    mask = ~np.isnan(val) & (val > 0)
    if mask.any():
        ax_pv.plot(sx[mask], val[mask], color=PALETTE["PPO"], lw=1.5, label="PPO")

# PPO policy gradient loss
if "PPO" in RESULTS:
    cb  = RESULTS["PPO"]["cb"]
    sx  = _steps(cb)
    val = _smooth(_get(cb, "policy_gradient_loss"))
    mask = ~np.isnan(val)
    ax_ppg.plot(sx[mask], val[mask], color=PALETTE["PPO"], lw=1.5, label="PPO")

_style_ax(ax_c,   "Critic Loss (SAC / TD3)",          ylabel="Loss")
_style_ax(ax_a,   "Actor Loss  (SAC / TD3)",           ylabel="Loss")
_style_ax(ax_pv,  "Value Loss  (PPO)",                 ylabel="Loss")
_style_ax(ax_ppg, "Policy Gradient Loss (PPO)",        ylabel="Loss")

for ax in (ax_c, ax_a, ax_pv, ax_ppg):
    ax.legend(fontsize=9, facecolor="#1e293b", edgecolor="#334155", labelcolor="white")

plt.tight_layout()
plt.show()

### Chart: Mean Q-Value · Entropy · Alpha (α)

In [ ]:
fig, axes = _fig(1, 3, figsize=(16, 5), title="Q-Value  ·  Entropy  ·  Alpha (α)")
ax_q, ax_ent, ax_alp = axes[0]

for name, res in RESULTS.items():
    cb  = res["cb"]
    sx  = _steps(cb)
    col = PALETTE[name]

    q   = _get(cb, "mean_q_value")
    ent = _get(cb, "entropy")
    alp = _get(cb, "ent_coef")

    mask_q   = ~np.isnan(q)
    mask_ent = ~np.isnan(ent)
    mask_alp = ~np.isnan(alp) & (alp > 0)

    if mask_q.any():
        ax_q.plot(sx[mask_q],   _smooth(q[mask_q]),   color=col, lw=1.6, label=name)
    if mask_ent.any():
        ax_ent.plot(sx[mask_ent], _smooth(ent[mask_ent]), color=col, lw=1.6, label=name)
    if mask_alp.any():
        ax_alp.plot(sx[mask_alp], _smooth(alp[mask_alp]), color=col, lw=1.6, label=name)

_style_ax(ax_q,   "Mean Q-Value (replay buffer sample)", ylabel="Q")
_style_ax(ax_ent, "Policy Entropy",                       ylabel="Entropy (nats)")
_style_ax(ax_alp, "Temperature α (ent_coef)",             ylabel="α")

for ax in (ax_q, ax_ent, ax_alp):
    ax.legend(fontsize=9, facecolor="#1e293b", edgecolor="#334155", labelcolor="white")

plt.tight_layout()
plt.show()

### Chart: Gradient Norms

In [ ]:
fig, axes = _fig(1, 3, figsize=(16, 5), title="Gradient Norms (L2)")
ax_ag, ax_cg, ax_pg = axes[0]

for name, res in RESULTS.items():
    cb  = res["cb"]
    sx  = _steps(cb)
    col = PALETTE[name]

    ag = _get(cb, "actor_grad_norm")
    cg = _get(cb, "critic_grad_norm")
    pg = _get(cb, "policy_grad_norm")      # PPO labels its network "policy"

    mask_ag = ~np.isnan(ag) & (ag > 0)
    mask_cg = ~np.isnan(cg) & (cg > 0)
    mask_pg = ~np.isnan(pg) & (pg > 0)

    if mask_ag.any():
        ax_ag.plot(sx[mask_ag], _smooth(ag[mask_ag]), color=col, lw=1.6, label=name)
    if mask_cg.any():
        ax_cg.plot(sx[mask_cg], _smooth(cg[mask_cg]), color=col, lw=1.6, label=name)
    if mask_pg.any():
        ax_pg.plot(sx[mask_pg], _smooth(pg[mask_pg]), color=col, lw=1.6, label=name)

_style_ax(ax_ag, "Actor  ∇ Norm (SAC / TD3)", ylabel="|∇|₂")
_style_ax(ax_cg, "Critic ∇ Norm (SAC / TD3)", ylabel="|∇|₂")
_style_ax(ax_pg, "Policy ∇ Norm (PPO)",        ylabel="|∇|₂")

for ax in (ax_ag, ax_cg, ax_pg):
    ax.legend(fontsize=9, facecolor="#1e293b", edgecolor="#334155", labelcolor="white")

plt.tight_layout()
plt.show()

### Chart: Replay Buffer Utilisation

In [ ]:
fig, axes = _fig(1, 2, figsize=(14, 5), title="Replay Buffer — Size & % Utilisation")
ax_sz, ax_pct = axes[0]

capacity_drawn = set()

for name in ("SAC", "TD3"):
    if name not in RESULTS: continue
    cb  = RESULTS[name]["cb"]
    sx  = _steps(cb)
    col = PALETTE[name]

    sz  = _get(cb, "replay_buffer_size")
    pct = _get(cb, "replay_buffer_pct")
    cap = _get(cb, "replay_buffer_capacity")

    mask = ~np.isnan(sz)
    if mask.any():
        ax_sz.plot(sx[mask], sz[mask] / 1000, color=col, lw=1.8, label=name)
        ax_pct.plot(sx[mask], pct[mask], color=col, lw=1.8, label=name)

    # Draw capacity line once per capacity value
    valid_cap = cap[mask]
    if len(valid_cap) > 0:
        cap_val = valid_cap[0]
        if cap_val not in capacity_drawn:
            ax_sz.axhline(cap_val / 1000, color=col, lw=1, ls="--", alpha=0.4,
                          label=f"{name} capacity")
            ax_pct.axhline(100, color="#ef4444", lw=1, ls="--", alpha=0.4)
            capacity_drawn.add(cap_val)

_style_ax(ax_sz,  "Buffer Size (k transitions)", ylabel="Size (×10³)")
_style_ax(ax_pct, "Buffer Utilisation (%)",       ylabel="% Full")

for ax in (ax_sz, ax_pct):
    ax.legend(fontsize=9, facecolor="#1e293b", edgecolor="#334155", labelcolor="white")

ax_pct.set_ylim(0, 110)
plt.tight_layout()
plt.show()

### Chart: Action Distribution (final snapshot)

Shows the mean ± std of each action dimension from the last 200 steps.

In [ ]:
n_algos  = len(RESULTS)
fig, axes = _fig(1, n_algos, figsize=(6 * n_algos, 5), title="Action Distribution — Final Policy")

short_names = [k.replace("_draw_temp", "").replace("_", "\n") for k in ACTION_KEYS]
x_pos = np.arange(len(ACTION_KEYS))

for col_idx, (name, res) in enumerate(RESULTS.items()):
    ax   = axes[0][col_idx]
    cb   = res["cb"]
    col  = PALETTE[name]

    # Find last snapshot that has action_distribution
    ad = None
    for snap in reversed(cb.history):
        if "action_distribution" in snap:
            ad = snap["action_distribution"]
            break

    if ad is None:
        ax.text(0.5, 0.5, "No action data", transform=ax.transAxes,
                ha="center", va="center", color="#94a3b8")
        _style_ax(ax, f"{name} — Action Distribution")
        continue

    means = np.array(ad["means"])
    stds  = np.array(ad["stds"])

    bars = ax.bar(x_pos, means, color=ACTION_COLORS[:len(x_pos)],
                  alpha=0.8, width=0.6, zorder=2)
    ax.errorbar(x_pos, means, yerr=stds, fmt="none",
                ecolor="white", elinewidth=1.5, capsize=4, zorder=3)

    # Reference line at 0 (normalised action space [-1, 1])
    ax.axhline(0, color="#475569", lw=0.8, ls="--", alpha=0.6)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(short_names, fontsize=8, color="#94a3b8")
    ax.set_ylim(-1.3, 1.3)
    _style_ax(ax, f"{name} — Action Mean ± Std", ylabel="Normalised action (−1 … 1)")

plt.tight_layout()
plt.show()

### Chart: PPO-Specific Metrics (KL divergence · Clip fraction)

In [ ]:
if "PPO" in RESULTS:
    cb  = RESULTS["PPO"]["cb"]
    sx  = _steps(cb)
    col = PALETTE["PPO"]

    fig, axes = _fig(1, 2, figsize=(13, 5), title="PPO — Approximate KL & Clip Fraction")
    ax_kl, ax_clip = axes[0]

    kl   = _get(cb, "approx_kl")
    clip = _get(cb, "clip_fraction")

    mask_kl   = ~np.isnan(kl)   & (kl   > 0)
    mask_clip = ~np.isnan(clip) & (clip > 0)

    if mask_kl.any():
        ax_kl.plot(sx[mask_kl], _smooth(kl[mask_kl]), color=col, lw=1.8, label="PPO")
        ax_kl.axhline(0.02, color="#ef4444", lw=1, ls="--", alpha=0.5,
                      label="KL threshold (0.02)")

    if mask_clip.any():
        ax_clip.plot(sx[mask_clip], _smooth(clip[mask_clip]), color=col, lw=1.8, label="PPO")

    _style_ax(ax_kl,   "Approx KL Divergence",    ylabel="KL")
    _style_ax(ax_clip, "Clip Fraction",             ylabel="Fraction clipped")

    for ax in (ax_kl, ax_clip):
        ax.legend(fontsize=9, facecolor="#1e293b", edgecolor="#334155", labelcolor="white")

    plt.tight_layout()
    plt.show()
else:
    print("PPO not trained — skipping this cell.")

---
## Section 5 — Full Dashboard (Single-Figure Summary)

One combined figure reproduces the full web dashboard layout for quick side-by-side comparison.

In [ ]:
## ── 18. Full Dashboard — All Metrics in One Figure ──────────────────────

fig = plt.figure(figsize=(20, 22), facecolor=FIG_BG)
fig.suptitle("ADU + NSU + VDU RL Agent Training Dashboard — Architecture Comparison",
             color="white", fontsize=15, y=0.99)

gs = gridspec.GridSpec(5, 3, figure=fig, hspace=0.52, wspace=0.32)

def sp(r, c):
    ax = fig.add_subplot(gs[r, c])
    ax.set_facecolor(PANEL_BG)
    return ax

# Row 0: Reward curves (full width)
ax_avg   = sp(0, 0)
ax_best  = sp(0, 1)
ax_ep    = sp(0, 2)

# Row 1: Losses
ax_crit  = sp(1, 0)
ax_act   = sp(1, 1)
ax_ppv   = sp(1, 2)

# Row 2: Q-value / Entropy / Alpha
ax_q     = sp(2, 0)
ax_ent   = sp(2, 1)
ax_alp   = sp(2, 2)

# Row 3: Gradient norms
ax_ag    = sp(3, 0)
ax_cg    = sp(3, 1)
ax_buf   = sp(3, 2)

# Row 4: Action distribution overlay (3 panels)
ax_ad0   = sp(4, 0)
ax_ad1   = sp(4, 1)
ax_ad2   = sp(4, 2)

action_axes = [ax_ad0, ax_ad1, ax_ad2]

short_names = [k.replace("_draw_temp","").replace("_","\n") for k in ACTION_KEYS]
x_pos = np.arange(len(ACTION_KEYS))

for algo_idx, (name, res) in enumerate(RESULTS.items()):
    cb  = res["cb"]
    sx  = _steps(cb)
    col = PALETTE[name]
    lw  = 1.6
    lbl = name

    # ── Reward ──────────────────────────────────────────────────────────
    avg  = _get(cb, "avg_reward")
    best = _get(cb, "best_reward")
    ax_avg.plot(sx, avg,  color=col, lw=lw, label=lbl)
    ax_best.plot(sx, best, color=col, lw=lw, label=lbl)
    ep   = _get(cb, "episode")
    ax_ep.plot(sx, ep, color=col, lw=lw, label=lbl)

    # ── Losses ───────────────────────────────────────────────────────────
    cr  = _smooth(_get(cb, "critic_loss"))
    ac  = _smooth(_get(cb, "actor_loss"))
    mask_cr = ~np.isnan(cr) & (cr > 0)
    mask_ac = ~np.isnan(ac)
    if mask_cr.any(): ax_crit.plot(sx[mask_cr], cr[mask_cr], color=col, lw=lw, label=lbl)
    if mask_ac.any(): ax_act.plot( sx[mask_ac], ac[mask_ac], color=col, lw=lw, label=lbl)

    vl  = _smooth(_get(cb, "value_loss"))
    mask_vl = ~np.isnan(vl) & (vl > 0)
    if mask_vl.any(): ax_ppv.plot(sx[mask_vl], vl[mask_vl], color=col, lw=lw, label=lbl)

    # ── Q / Entropy / Alpha ──────────────────────────────────────────────
    qv  = _smooth(_get(cb, "mean_q_value"))
    ent = _smooth(_get(cb, "entropy"))
    alp = _smooth(_get(cb, "ent_coef"))
    mask_q   = ~np.isnan(qv)
    mask_ent = ~np.isnan(ent)
    mask_alp = ~np.isnan(alp) & (alp > 0)
    if mask_q.any():   ax_q.plot(  sx[mask_q],   qv[mask_q],   color=col, lw=lw, label=lbl)
    if mask_ent.any(): ax_ent.plot(sx[mask_ent], ent[mask_ent], color=col, lw=lw, label=lbl)
    if mask_alp.any(): ax_alp.plot(sx[mask_alp], alp[mask_alp], color=col, lw=lw, label=lbl)

    # ── Gradient norms ───────────────────────────────────────────────────
    ag  = _smooth(_get(cb, "actor_grad_norm"))
    cg  = _smooth(_get(cb, "critic_grad_norm"))
    pg  = _smooth(_get(cb, "policy_grad_norm"))
    mask_ag = ~np.isnan(ag) & (ag > 0)
    mask_cg = ~np.isnan(cg) & (cg > 0)
    mask_pg = ~np.isnan(pg) & (pg > 0)
    if mask_ag.any(): ax_ag.plot(sx[mask_ag], ag[mask_ag], color=col, lw=lw, label=lbl)
    if mask_cg.any(): ax_cg.plot(sx[mask_cg], cg[mask_cg], color=col, lw=lw, label=lbl)
    if mask_pg.any(): ax_cg.plot(sx[mask_pg], pg[mask_pg], color=col, lw=lw, ls="--", label=f"{lbl} (policy)")

    # ── Buffer utilisation ────────────────────────────────────────────────
    buf = _get(cb, "replay_buffer_pct")
    mask_buf = ~np.isnan(buf)
    if mask_buf.any(): ax_buf.plot(sx[mask_buf], buf[mask_buf], color=col, lw=lw, label=lbl)

    # ── Action distribution ───────────────────────────────────────────────
    ad = None
    for snap in reversed(cb.history):
        if "action_distribution" in snap:
            ad = snap["action_distribution"]
            break

    if algo_idx < len(action_axes) and ad:
        ax_ad = action_axes[algo_idx]
        means = np.array(ad["means"])
        stds  = np.array(ad["stds"])
        ax_ad.bar(x_pos, means, color=ACTION_COLORS[:len(x_pos)],
                  alpha=0.8, width=0.6, zorder=2)
        ax_ad.errorbar(x_pos, means, yerr=stds, fmt="none",
                       ecolor="white", elinewidth=1.2, capsize=3, zorder=3)
        ax_ad.axhline(0, color="#475569", lw=0.8, ls="--", alpha=0.6)
        ax_ad.set_xticks(x_pos)
        ax_ad.set_xticklabels(short_names, fontsize=7, color="#94a3b8")
        ax_ad.set_ylim(-1.35, 1.35)
        _style_ax(ax_ad, f"{name} — Action Distribution", ylabel="Norm. action")

# Styling
_style_ax(ax_avg,  "Avg Reward (100-ep)",        ylabel="Reward")
_style_ax(ax_best, "Best Reward",                 ylabel="Reward")
_style_ax(ax_ep,   "Episode Count",               ylabel="Episodes")
_style_ax(ax_crit, "Critic Loss  (off-policy)",   ylabel="Loss")
_style_ax(ax_act,  "Actor Loss   (off-policy)",   ylabel="Loss")
_style_ax(ax_ppv,  "Value Loss   (PPO)",          ylabel="Loss")
_style_ax(ax_q,    "Mean Q-Value",                ylabel="Q")
_style_ax(ax_ent,  "Policy Entropy",              ylabel="nats")
_style_ax(ax_alp,  "Temperature α",              ylabel="α")
_style_ax(ax_ag,   "Actor ∇ Norm",               ylabel="|∇|₂")
_style_ax(ax_cg,   "Critic / Policy ∇ Norm",     ylabel="|∇|₂")
_style_ax(ax_buf,  "Replay Buffer Utilisation",   ylabel="% Full")

for ax in (ax_avg, ax_best, ax_ep, ax_crit, ax_act, ax_ppv,
           ax_q, ax_ent, ax_alp, ax_ag, ax_cg, ax_buf):
    ax.legend(fontsize=8, facecolor="#1e293b", edgecolor="#334155",
              labelcolor="white", loc="best")

plt.savefig(PROJECT_ROOT / "RL_agent" / "dashboard.png", dpi=120,
            bbox_inches="tight", facecolor=FIG_BG)
print("Dashboard saved → RL_agent/dashboard.png")
plt.show()

---
## Section 6 — Summary Table & Verdict

Numeric comparison of all architectures at the end of training.

In [ ]:
## ── 19. Summary Comparison Table ────────────────────────────────────────

rows = []
for name, res in RESULTS.items():
    cb   = res["cb"]
    t    = res["time"]
    last = cb.history[-1] if cb.history else {}

    rows.append({
        "Algorithm":            name,
        "Train time (s)":       round(t, 1),
        "Episodes":             cb.episode_count,
        "Best reward":          round(cb.best_reward, 4),
        "Avg reward (100-ep)":  round(float(np.mean(cb.episode_rewards[-100:]))
                                      if cb.episode_rewards else 0.0, 4),
        "Critic loss":          round(last.get("critic_loss", float("nan")), 5),
        "Actor loss":           round(last.get("actor_loss",  float("nan")), 4),
        "Mean Q-value":         round(last.get("mean_q_value",float("nan")), 4),
        "Entropy":              round(last.get("entropy",     float("nan")), 4),
        "Alpha (α)":            round(last.get("ent_coef",    float("nan")), 5),
        "Actor ∇ norm":        round(last.get("actor_grad_norm",  float("nan")), 4),
        "Critic ∇ norm":       round(last.get("critic_grad_norm", float("nan")), 4),
        "Buffer fill (%)":      last.get("replay_buffer_pct", float("nan")),
    })

df = pd.DataFrame(rows).set_index("Algorithm")
print(df.to_string())
print()
print("─" * 70)

# Find best on avg reward
if not df["Avg reward (100-ep)"].isna().all():
    winner = df["Avg reward (100-ep)"].idxmax()
    wr     = df.loc[winner, "Avg reward (100-ep)"]
    print(f"\n🏆  Best avg reward: {winner} ({wr:.4f})")

# Find fastest
fastest = df["Train time (s)"].idxmin()
print(f"⚡  Fastest training: {fastest} ({df.loc[fastest,'Train time (s)']:.1f}s)")

In [ ]:
## ── 20. Bar Chart — Final Metric Scorecard ───────────────────────────────

metrics_to_plot = [
    ("Best reward",        "Best\nReward",   1),
    ("Avg reward (100-ep)","Avg\nReward",    1),
    ("Critic loss",        "Critic\nLoss",  -1),  # lower = better
    ("Actor loss",         "Actor\nLoss",   -1),
    ("Entropy",            "Entropy",        1),
]

fig, axes = _fig(1, len(metrics_to_plot), figsize=(16, 5),
                 title="Final Scorecard — Key Metrics per Architecture")

names = list(RESULTS.keys())
colors = [PALETTE[n] for n in names]
x = np.arange(len(names))

for idx, (col_key, col_label, direction) in enumerate(metrics_to_plot):
    ax = axes[0][idx]
    vals = [df.loc[n, col_key] if n in df.index else float("nan") for n in names]

    # Normalise to [0,1] range for visual comparison
    arr = np.array(vals, dtype=float)
    valid = arr[~np.isnan(arr)]
    if len(valid) > 0:
        lo, hi = valid.min(), valid.max()
        rng = hi - lo if hi != lo else 1.0
        norm = (arr - lo) / rng * direction  # flip sign for "lower is better"
    else:
        norm = arr

    bars = ax.bar(x, arr, color=colors, alpha=0.85, width=0.55, zorder=2)

    # Annotate bar tops
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + (abs(bar.get_height()) * 0.04 + 1e-8),
                    f"{v:.3f}", ha="center", va="bottom",
                    color="white", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(names, color="#94a3b8", fontsize=9)
    _style_ax(ax, col_label, xlabel="")

plt.tight_layout()
plt.show()

---
## Section 7 — DWSIM Live Inference Test

Run one episode of the best-performing model against the **real DWSIM flowsheet**.  
Requires `USE_DWSIM = True` and a running DWSIM installation.

In [ ]:
## ── 21. DWSIM Live Inference ──────────────────────────────────────────────
# Pick the algorithm with the best avg reward, run one episode on DWSIM.

if USE_DWSIM and RESULTS:
    # Choose best architecture
    best_algo = max(RESULTS, key=lambda n: (
        float(np.mean(RESULTS[n]["cb"].episode_rewards[-100:]))
        if RESULTS[n]["cb"].episode_rewards else -9e9
    ))
    model = RESULTS[best_algo]["model"]
    print(f"Running live DWSIM episode with: {best_algo}")

    env = make_env(use_mock=False)
    obs, _ = env.reset()

    episode_reward = 0.0
    step_log = []
    done = False

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward
        done = terminated or truncated

        from backend.core.rl_environment import ACTION_SCALES, ACTION_KEYS
        real_action = action * np.array([ACTION_SCALES[k] for k in ACTION_KEYS])
        row = {"step": info["step"], "reward": round(reward, 5)}
        for k, v in zip(ACTION_KEYS, real_action):
            row[k] = round(float(v), 2)
        step_log.append(row)

    env.close()

    df_log = pd.DataFrame(step_log)
    print(f"\nEpisode complete: {len(step_log)} steps | Total reward: {episode_reward:.4f}")
    print("\nLast 5 steps:")
    print(df_log.tail().to_string(index=False))

    # Plot actions taken during the episode
    fig, axes = _fig(1, 2, figsize=(14, 5), title=f"{best_algo} — Live DWSIM Episode")
    ax_r, ax_a = axes[0]

    ax_r.plot(df_log["step"], df_log["reward"], color=PALETTE[best_algo], lw=1.8)
    ax_r.fill_between(df_log["step"], df_log["reward"], alpha=0.15, color=PALETTE[best_algo])
    _style_ax(ax_r, "Step Reward (DWSIM)", ylabel="Reward")

    for i, key in enumerate(ACTION_KEYS):
        if key in df_log.columns:
            ax_a.plot(df_log["step"], df_log[key],
                      color=ACTION_COLORS[i], lw=1.2,
                      label=key.replace("_draw_temp","").replace("_"," "))

    _style_ax(ax_a, "Actions Applied to Column", ylabel="Real-unit value")
    ax_a.legend(fontsize=8, facecolor="#1e293b", edgecolor="#334155", labelcolor="white",
                bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

else:
    print("DWSIM inference skipped (USE_DWSIM=False).")
    print("Set USE_DWSIM=True and re-run cells 1 & 3, then re-run this cell.")

In [ ]:
## ── 22. Save Models & Metrics (local + Firebase) ─────────────────────────

import asyncio
from backend.services.firebase_service import FirebaseService

firebase = FirebaseService()

SAVE_DIR = PROJECT_ROOT / "checkpoints" / "notebook"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")

for name, res in RESULTS.items():
    cb    = res["cb"]
    model = res["model"]

    run_id = f"notebook_{name}_{ts}"

    # ── 1. Save SB3 model locally ──────────────────────────────────────
    save_path = SAVE_DIR / run_id
    model.save(str(save_path))
    zip_path = str(save_path) + ".zip"

    # ── 2. Save metrics history locally ────────────────────────────────
    metrics_dict = {
        "algorithm":    name,
        "timestamp":    ts,
        "episodes":     cb.episode_count,
        "best_reward":  float(cb.best_reward),
        "avg_reward":   float(np.mean(cb.episode_rewards[-100:]))
                        if cb.episode_rewards else 0.0,
        "history":      cb.history,
    }
    metrics_path = SAVE_DIR / f"{run_id}_metrics.json"
    with open(metrics_path, "w") as f:
        json.dump(metrics_dict, f, indent=2)

    print(f"  {name:5s}  → {zip_path}")
    print(f"         metrics → {metrics_path}")

    # ── 3. Persist to Firebase ─────────────────────────────────────────
    try:
        loop = asyncio.new_event_loop()

        # Training run record
        loop.run_until_complete(firebase.save_training_run({
            "run_id":       run_id,
            "algorithm":    name,
            "source":       "notebook",
            "total_timesteps": TOTAL_TIMESTEPS,
            "best_reward":  metrics_dict["best_reward"],
            "avg_reward":   metrics_dict["avg_reward"],
            "episodes":     metrics_dict["episodes"],
        }))

        # Training metrics (history limited to last 200 points for Firestore 1MB cap)
        loop.run_until_complete(firebase.save_training_metrics(
            run_id,
            final_metrics=metrics_dict,
            metrics_history=cb.history[-200:] if cb.history else [],
        ))

        # Checkpoint (.zip → Firebase Storage + metadata → Firestore)
        loop.run_until_complete(firebase.save_checkpoint(
            run_id=run_id,
            local_zip_path=zip_path,
            metadata={
                "algorithm":    name,
                "source":       "notebook",
                "best_reward":  metrics_dict["best_reward"],
                "avg_reward":   metrics_dict["avg_reward"],
                "episodes":     metrics_dict["episodes"],
                "total_timesteps": TOTAL_TIMESTEPS,
            },
        ))

        loop.close()
        print(f"         ☁  Firebase: training run + metrics + checkpoint uploaded")
    except Exception as exc:
        print(f"         ⚠  Firebase save failed (local files kept): {exc}")

print(f"\n All models + metrics saved to {SAVE_DIR}")
print(" Firebase persistence attempted for all agents")

---
## Conclusions & Architecture Guide

### When to use each algorithm for ADU + NSU + VDU optimisation

| Algorithm | Strengths | Limitations | Best for ADU+NSU+VDU use-case |
|---|---|---|---|
| **SAC** | Max-entropy → robust exploration; automatic α tuning; sample-efficient | Needs replay buffer memory; sensitive to reward scale | **Default choice** — handles continuous 10-dim action space well |
| **TD3** | Deterministic policy → fine-grained control; stable Q-learning | No entropy bonus; greedy in exploration; can get stuck | Good when a near-optimal starting policy is already known |
| **PPO** | On-policy → no stale data issues; very stable training | Less sample-efficient; no Q estimates; requires more wall-time | Useful for safety-critical re-training from a good checkpoint |

### Product streams (13 total)

| Column | Products |
|---|---|
| **ADU** (Atmospheric) | Uncondensed Gas, Heavy Naphtha, SKO (Jet Fuel), Light Gas Oil, Heavy Gas Oil |
| **NSU** (Naphtha Stabilizer) | StabOffGas, LPG, SRN (Straight-Run Naphtha) |
| **VDU** (Vacuum) | Offgas, Vacuum Diesel, Vacuum Gas Oil, Hotwell Oil, Vac Residue |

### D95% Quality Constraint

D95% (95% distillation temperature) is estimated from component compositions and Normal Boiling Points for each product stream. Products exceeding their D95% specification incur a penalty in the reward function.

### Metric interpretation

| Metric | Healthy sign | Warning sign |
|---|---|---|
| Avg reward | Increasing trend | Flat or decreasing after many updates |
| Critic loss | Decreasing, then stable | Diverging (exploding) |
| Actor loss | Stable or slowly rising | Oscillating wildly |
| Mean Q-value | Negative (rewards are scaled /1000) | Large positive → Q overestimation |
| Entropy | Slowly decreasing over training | Collapsed to 0 early → premature convergence |
| α (ent_coef) | Decreasing as policy sharpens | Stuck high → policy not converging |
| Gradient norms | < 1.0 for actor, < 50 for critic | Exploding (> 100) → clip or lower LR |
| Buffer fill | Reaches 100% after ~buffer_size steps | Never fills → training too fast / env too slow |

### Next steps
1. Increase `TOTAL_TIMESTEPS` to 200 000+ for production-quality policies  
2. Set `USE_DWSIM = True` to train on the real flowsheet (`main_sim.dwxmz`)  
3. Enable curriculum learning by gradually increasing `curriculum_level`  
4. Tune reward scaling (currently `/1000`) if critic loss diverges  
5. Load the best checkpoint into the web application via the Checkpoints panel